# COMP9312 Project Q2: k-triangle core index


Run the cells from top to bottom. Only edit the `KTriangleIndex` code cell.


## 1. Graph Data Structure
The following code defines the input graph representation. Run this cell first. Do not edit this cell.


In [ ]:
class Graph:
    def __init__(self, n, edges):
        """
        Build an undirected graph using a compact adjacency-array structure.

        Parameters
        ----------
        n : int
            Number of vertices. Vertices are assumed to be numbered from 0 to n - 1.

        edges : list[tuple[int, int]]
            List of undirected edges. For example, (u, v) means there is an edge
            between vertex u and vertex v.
        """

        # Number of vertices
        self.n = n

        # Number of undirected edges
        self.m = len(edges)

        # degree[v] stores the degree of vertex v
        degree = [0] * n

        # Count the degree of each vertex
        for u, v in edges:
            degree[u] += 1
            degree[v] += 1

        # offsets[v] stores the starting position of vertex v's neighbors
        # in the indices array.
        #
        # The neighbors of vertex v are stored in:
        #
        # indices[offsets[v] : offsets[v + 1]]
        self.offsets = [0] * (n + 1)

        # Build prefix sums from the degree array
        for v in range(n):
            self.offsets[v + 1] = self.offsets[v] + degree[v]

        # indices stores all adjacency lists in one flat array.
        # Since the graph is undirected, each edge is stored twice:
        # u -> v and v -> u.
        self.indices = [0] * (2 * self.m)

        # cursor[v] points to the next free position in vertex v's adjacency area
        cursor = self.offsets[:-1].copy()

        # Fill the adjacency array
        for u, v in edges:
            self.indices[cursor[u]] = v
            cursor[u] += 1

            self.indices[cursor[v]] = u
            cursor[v] += 1

        # Sort each vertex's adjacency list.
        # This makes edge lookup efficient using binary search.
        for v in range(n):
            start = self.offsets[v]
            end = self.offsets[v + 1]
            self.indices[start:end] = sorted(self.indices[start:end])

    def neighbors(self, v):
        """
        Return all neighbors of vertex v.

        Example
        -------
        If vertex 1 is connected to 0, 2, and 3,
        then neighbors(1) returns [0, 2, 3].
        """

        start = self.offsets[v]
        end = self.offsets[v + 1]
        return self.indices[start:end]

    def degree(self, v):
        """
        Return the degree of vertex v.

        Example
        -------
        If vertex 1 has three neighbors,
        then degree(1) returns 3.
        """

        return self.offsets[v + 1] - self.offsets[v]

    def has_edge(self, u, v):
        """
        Return True if there is an edge between u and v.

        The adjacency list of u is sorted, so binary search can be used.

        Example
        -------
        If the graph contains edge (0, 2), then has_edge(0, 2) returns True.
        If the graph does not contain edge (0, 4), then has_edge(0, 4) returns False.
        """

        left = self.offsets[u]
        right = self.offsets[u + 1] - 1

        while left <= right:
            mid = (left + right) // 2

            if self.indices[mid] == v:
                return True
            elif self.indices[mid] < v:
                left = mid + 1
            else:
                right = mid - 1

        return False

## 2. Code Template
Only edit this cell. Implement `KTriangleIndex.build()` and `KTriangleIndex.query(k, v)`.


In [ ]:
################################################################################
# Import Python Standard Library modules here if needed.
from typing import List
################################################################################


class KTriangleIndex:
    """K-triangle core index.

    The input graph is available as self.graph (a Graph object).

    Design overview
    ---------------
    A vertex's *triangle support* is the number of triangles it lies in inside a
    subgraph. The k-triangle core is the vertex analogue of the k-core: replace
    "degree" by "number of triangles". We compute, for every vertex, its
    triangle-core number tc(v) = the largest k for which v belongs to a
    k-triangle core, using a peeling (degeneracy) process on triangle counts.

    Connectivity of a k-triangle core is *triangle connectivity*: two vertices
    are in the same k-triangle core iff they are linked by a chain of triangles
    (sharing vertices), every vertex of which has tc >= k. We capture this with
    an edge level:

        level(u, w) = max over triangles (u, w, x) of min(tc(u), tc(w), tc(x)),

    i.e. the highest k at which edge (u, w) sits inside a triangle whose three
    vertices all survive to level k. An edge in no triangle has level 0.

    A query(k, v) is then: if tc(v) < k return []; otherwise return the vertices
    reachable from v using only edges with level >= k. That set is exactly the
    triangle-connected k-core component containing v.

    Index (built once in build()):
      * self.tc          : tc(v) for every vertex.
      * self.edge_level  : level of each incident edge, laid out parallel to
                           graph.indices so a query is a plain BFS/DFS.
    """

    def __init__(self, graph: "Graph"):
        self.graph = graph
        self.tc = None          # tc[v] = triangle-core number of vertex v
        self.edge_level = None  # parallel to graph.indices: level of each incident edge

    # ------------------------------------------------------------------ build
    def build(self) -> None:
        """Build the k-triangle core index."""
        g = self.graph
        n = g.n
        offsets = g.offsets
        indices = g.indices

        deg = [offsets[v + 1] - offsets[v] for v in range(n)]

        # --- 1. List every triangle exactly once ---------------------------
        # Orient each edge from the lower-rank to the higher-rank endpoint,
        # rank = (degree, id). Each triangle is then discovered once, from its
        # lowest-rank vertex. Total cost O(sum_edge min(d(u), d(w))) <= O(m^1.5).
        fwd = [[] for _ in range(n)]
        for u in range(n):
            ku = (deg[u], u)
            for i in range(offsets[u], offsets[u + 1]):
                w = indices[i]
                if ku < (deg[w], w):
                    fwd[u].append(w)

        triangles = []          # each triangle stored once as a (u, w, x) tuple
        tri_count = [0] * n     # tri_count[v] = number of triangles containing v
        marker = [-1] * n
        for u in range(n):
            for w in fwd[u]:
                marker[w] = u
            for w in fwd[u]:
                for x in fwd[w]:
                    if marker[x] == u:      # edge (u, x) exists -> triangle u-w-x
                        triangles.append((u, w, x))
                        tri_count[u] += 1
                        tri_count[w] += 1
                        tri_count[x] += 1

        T = len(triangles)

        # Triangles incident to each vertex (indices into `triangles`).
        vtri = [[] for _ in range(n)]
        for ti, (a, b, c) in enumerate(triangles):
            vtri[a].append(ti)
            vtri[b].append(ti)
            vtri[c].append(ti)

        # --- 2. Triangle-core peeling (Batagelj-Zaversnik on triangle counts) -
        # Repeatedly remove the vertex with the fewest remaining triangles.
        # Removing a vertex destroys all triangles through it, decrementing the
        # triangle counts of the other two vertices. tc(v) = the count value at
        # which v is removed (kept monotone by the bucket order). O(n + T) time.
        d = tri_count[:]                    # current triangle counts
        md = max(d) if n else 0

        # Bucket-sort vertices by current count into `vert`, with position map.
        bin_ = [0] * (md + 2)
        for v in range(n):
            bin_[d[v]] += 1
        start = 0
        for i in range(md + 1):
            bin_[i], start = start, start + bin_[i]
        vert = [0] * n                      # vertices ordered by current count
        pos = [0] * n                       # pos[v] = index of v inside vert
        for v in range(n):
            pos[v] = bin_[d[v]]
            vert[pos[v]] = v
            bin_[d[v]] += 1
        for i in range(md, 0, -1):          # restore bucket start pointers
            bin_[i] = bin_[i - 1]
        bin_[0] = 0

        alive_tri = bytearray([1]) * T      # 1 while a triangle still exists
        tc = [0] * n

        for idx in range(n):
            v = vert[idx]
            level = d[v]                     # smallest remaining count
            tc[v] = level                    # v's triangle-core number
            for ti in vtri[v]:
                if not alive_tri[ti]:
                    continue
                alive_tri[ti] = 0            # this triangle is destroyed with v
                a, b, c = triangles[ti]
                for w in (a, b, c):
                    dw = d[w]
                    # Only vertices still strictly above the current level move;
                    # already-finalised vertices (dw <= level) must not be touched.
                    if w == v or dw <= level:
                        continue
                    # Move w one bucket down: swap it with the first vertex of
                    # its bucket, then shrink the bucket.
                    pw = pos[w]
                    ps = bin_[dw]
                    fw = vert[ps]
                    if fw != w:
                        vert[pw] = fw
                        vert[ps] = w
                        pos[fw] = pw
                        pos[w] = ps
                    bin_[dw] += 1
                    d[w] = dw - 1

        self.tc = tc

        # --- 3. Edge levels ------------------------------------------------
        # For each edge, the max triangle level over triangles containing it.
        best = {}
        for (a, b, c) in triangles:
            level = tc[a]
            if tc[b] < level:
                level = tc[b]
            if tc[c] < level:
                level = tc[c]
            for (x, y) in ((a, b), (a, c), (b, c)):
                key = (x, y) if x < y else (y, x)
                if best.get(key, -1) < level:
                    best[key] = level

        # Scatter the undirected edge levels into an array parallel to
        # graph.indices so query() is a simple threshold traversal.
        edge_level = [0] * (2 * g.m)
        for u in range(n):
            for i in range(offsets[u], offsets[u + 1]):
                w = indices[i]
                key = (u, w) if u < w else (w, u)
                lv = best.get(key)
                if lv is not None:
                    edge_level[i] = lv
        self.edge_level = edge_level

    # ------------------------------------------------------------------ query
    def query(self, k: int, v: int) -> List[int]:
        """Return the vertices of the k-triangle core containing v."""
        if self.tc[v] < k:
            return []

        offsets = self.graph.offsets
        indices = self.graph.indices
        edge_level = self.edge_level

        # DFS over edges whose level >= k (their endpoints all have tc >= k).
        visited = {v}
        stack = [v]
        while stack:
            x = stack.pop()
            for i in range(offsets[x], offsets[x + 1]):
                if edge_level[i] >= k:
                    w = indices[i]
                    if w not in visited:
                        visited.add(w)
                        stack.append(w)
        return list(visited)

## 3. How to Test Your Code
The following tests use the `KTriangleIndex` class defined above, generate fixed-seed graphs, and compare your output with precomputed answers. The largest test has 10,000 vertices and 80,000 edges.


In [ ]:
################################################################################
# Do not edit this code cell.
import time
from itertools import accumulate, combinations
################################################################################


def generate_test_graph(seed, num_cliques=4):
    rng = _LCG(seed)
    clique_sizes = [rng.randint(4, 6) for _ in range(num_cliques)]
    prefix = [0] + list(accumulate(clique_sizes))
    edges = set()

    for idx, size in enumerate(clique_sizes):
        base = prefix[idx]
        for i, j in combinations(range(size), 2):
            edges.add((base + i, base + j))

    for idx in range(num_cliques - 1):
        u = prefix[idx] + clique_sizes[idx] - 1
        v = prefix[idx + 1]
        if u > v:
            u, v = v, u
        edges.add((u, v))

    next_id = prefix[-1]
    for _ in range(num_cliques):
        clique_id = rng.randint(0, num_cliques - 1)
        anchor = prefix[clique_id] + rng.randint(0, clique_sizes[clique_id] - 1)
        a, b = next_id, next_id + 1
        next_id += 2
        for u, v in ((anchor, a), (anchor, b), (a, b)):
            if u > v:
                u, v = v, u
            edges.add((u, v))

    return Graph(next_id, sorted(edges))


def generate_benchmark_graph(seed=9312):
    n = 10000
    edges = set()

    for i, j in combinations(range(0, 6), 2):
        edges.add((i, j))
    for i, j in combinations(range(6, 11), 2):
        edges.add((i, j))
    edges.add((5, 6))
    edges.add((10, 11))

    rng = _LCG(seed)
    left_start, left_end = 11, 5006
    right_start, right_end = 5006, 10000
    right_count = right_end - right_start

    for offset, u in enumerate(range(left_start, left_end)):
        shift = rng.randint(0, right_count - 1)
        for j in range(16):
            v = right_start + ((offset + shift + j) % right_count)
            edges.add((u, v))

    extra_edges_needed = 80000 - len(edges)
    added = 0
    cursor = 0
    while added < extra_edges_needed:
        u = left_start + (cursor % (left_end - left_start))
        v = right_start + ((cursor * 37 + 16) % right_count)
        edge = (u, v)
        if edge not in edges:
            edges.add(edge)
            added += 1
        cursor += 1

    return Graph(n, sorted(edges))


class _LCG:
    _a = 6364136223846793005
    _c = 1442695040888963407
    _m = 2 ** 64

    def __init__(self, seed=42):
        self.state = seed & 0xFFFFFFFFFFFFFFFF

    def rand64(self):
        self.state = (self._a * self.state + self._c) % self._m
        return self.state

    def randint(self, lo, hi):
        return lo + self.rand64() % (hi - lo + 1)


def run_tests() -> None:
    test_cases = [
        {
            "seed": 7,
            "generator": "small",
            "n": 30,
            "m": 66,
            "queries": [(1, 0), (3, 0), (3, 6), (4, 0), (1, 29), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4, 5, 24, 25, 28, 29],
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9],
                [0, 1, 2, 3, 4, 5],
                [0, 1, 2, 3, 4, 5, 24, 25, 28, 29],
                [0, 1, 2, 3, 4, 5],
            ],
        },
        {
            "seed": 42,
            "generator": "small",
            "n": 31,
            "m": 70,
            "queries": [(1, 0), (3, 0), (3, 6), (4, 0), (1, 30), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4, 5],
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9, 10, 11],
                [0, 1, 2, 3, 4, 5],
                [18, 19, 20, 21, 22, 25, 26, 29, 30],
                [0, 1, 2, 3, 4, 5],
            ],
        },
        {
            "seed": 2026,
            "generator": "small",
            "n": 28,
            "m": 56,
            "queries": [(1, 0), (3, 0), (3, 5), (4, 0), (1, 27), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4],
                [0, 1, 2, 3, 4],
                [5, 6, 7, 8, 9],
                [0, 1, 2, 3, 4],
                [16, 17, 18, 19, 22, 23, 26, 27],
                [],
            ],
        },
        {
            "seed": 9312,
            "generator": "benchmark",
            "n": 10000,
            "m": 80000,
            "queries": [(1, 0), (1, 7), (6, 7), (7, 7), (10, 0), (1, 11), (1, 9999)],
            "answers": [
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9, 10],
                [6, 7, 8, 9, 10],
                [],
                [0, 1, 2, 3, 4, 5],
                [],
                [],
            ],
        },
    ]

    for case in test_cases:
        if case["generator"] == "benchmark":
            graph = generate_benchmark_graph(case["seed"])
        else:
            graph = generate_test_graph(case["seed"])

        print(f"\nTesting {case['generator']} seed={case['seed']} | n={graph.n}, m={graph.m}")
        if graph.n != case["n"] or graph.m != case["m"]:
            print(f"Graph size mismatch: expected n={case['n']}, m={case['m']}")
            continue

        index = KTriangleIndex(graph)
        build_start = time.perf_counter()
        index.build()
        build_time = time.perf_counter() - build_start
        print(f"build time: {build_time:.6f} seconds")

        all_correct = True
        query_time_total = 0.0
        for query, expected in zip(case["queries"], case["answers"]):
            k, v = query
            query_start = time.perf_counter()
            actual = sorted(index.query(k, v))
            query_time = time.perf_counter() - query_start
            query_time_total += query_time
            ok = actual == expected
            all_correct = all_correct and ok
            status = "CORRECT" if ok else "INCORRECT"
            print(f"query({k}, {v}) -> {actual} | expected {expected} | {status} | time {query_time:.6f}s")
        print(f"total query time: {query_time_total:.6f} seconds")
        print("Result:", "CORRECT" if all_correct else "INCORRECT")

run_tests()
